# Quickstart to demonstrate an OCP agent

In [ ]:
!pip install deepagents langchain_openai langchain_mcp_adapters

## Create MCP tool Client

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient  
import os

In [ ]:
ocp_mcp_endpoint = os.environ["OCP_MCP_ENDPOINT"]

mcp_servers = {
        "openshift-mcp": {
            "transport": "http",
            "url": ocp_mcp_endpoint,
        }
    }

print(mcp_servers)

mcp_client = MultiServerMCPClient(mcp_servers)

tools = await mcp_client.get_tools()    

#print(tools)

# Create a connection to our LLM

In [ ]:
from langchain_openai import ChatOpenAI

qwen = ChatOpenAI(
    base_url=os.environ["QWEN_ENDPOINT"],
    api_key=os.environ["QWEN_API_KEY"],
    model="Qwen3.6-35B-A3B",
)

## let's create a subagent that specialises in OCP stuff

In [ ]:
ocp_prompt = """You are an expert on Openshift and Kubernetes. Your job is to gather information about a connected OpenShift cluster. Use the MCP tools at your disposal to accomplish this
"""

In [ ]:
from deepagents import AsyncSubAgent

ocp_subagent = AsyncSubAgent(
    name="ocp-agent",
    description="Used to report back on the state and to investigate a connected OCP cluster",
    system_prompt=ocp_prompt,
    tools=tools)

    

# let's create the main agent

In [ ]:
master_prompt = """You are a general purpose agent. You have access to a subagent that specialises in Openshift and Kubernetes and is connected to a live Openshift cluster"""

In [ ]:
from deepagents import create_deep_agent

subagents = [ocp_subagent]

agent = create_deep_agent(
    model=qwen,
    subagents=subagents,
    system_prompt=master_prompt,
)

# Test the agent

In [ ]:
result = await agent.ainvoke({"messages": [{"role": "user", "content": "List the pods currently running in the ocp-mcp-server namespace"}]})

# Print the agent's response
print(result["messages"][-1].content)